# InspecSafe Colab Minimal Validation

This notebook uses fixed cell names so we can debug it together reliably.

Run order:
1. Cell 1
2. Cell 2
3. Test Cell 3
4. Cell 4
5. Cell 5
6. Cell 6
7. Test Cell 7
8. Cell 8
9. Cell 9
10. Cell 10


In [ ]:
# Cell 1: GPU, base environment, and transformers pinning

!nvidia-smi
!python --version
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cuda_available', torch.cuda.is_available())"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("This is not a GPU runtime. In Colab, go to Runtime -> Change runtime type -> select T4 GPU, then reconnect and rerun.")

%env AM_I_DOCKER=False
%env BUILD_WITH_CUDA=True
%env CUDA_HOME=/usr/local/cuda

%cd /content

!python -m pip uninstall -y transformers tokenizers
!python -m pip install ninja
!python -m pip install opencv-python pillow pycocotools supervision "tokenizers==0.19.1" "transformers==4.41.2"

!rm -rf /content/GroundingDINO /content/segment-anything
!git clone https://github.com/IDEA-Research/GroundingDINO.git
!git clone https://github.com/facebookresearch/segment-anything.git


In [ ]:
# Cell 2: Patch GroundingDINO and install dependencies

%cd /content/GroundingDINO

from pathlib import Path
import os

cuda_file = Path("groundingdino/models/GroundingDINO/csrc/MsDeformAttn/ms_deform_attn_cuda.cu")
text = cuda_file.read_text(encoding="utf-8")

text = text.replace(
    'AT_DISPATCH_FLOATING_TYPES(value.type(), "ms_deform_attn_forward_cuda", ([&] {',
    'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(), "ms_deform_attn_forward_cuda", ([&] {'
)
text = text.replace(
    'AT_DISPATCH_FLOATING_TYPES(value.type(), "ms_deform_attn_backward_cuda", ([&] {',
    'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(), "ms_deform_attn_backward_cuda", ([&] {'
)

cuda_file.write_text(text, encoding="utf-8")
print("Patched:", cuda_file)

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["BUILD_WITH_CUDA"] = "True"
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

%cd /content/segment-anything
!python -m pip install -e .

%cd /content/GroundingDINO
!python -m pip install --no-build-isolation -e .


In [ ]:
# Test Cell 3: Verify pinned transformers and imports

import sys
from pathlib import Path

segment_anything_repo = Path("/content/segment-anything")
groundingdino_repo = Path("/content/GroundingDINO")

if str(segment_anything_repo) not in sys.path:
    sys.path.insert(0, str(segment_anything_repo))
if str(groundingdino_repo) not in sys.path:
    sys.path.insert(0, str(groundingdino_repo))

sys.modules.pop("transformers", None)

import transformers
print("transformers version =", transformers.__version__)
if transformers.__version__ != "4.41.2":
    raise RuntimeError(f"Expected transformers 4.41.2, got {transformers.__version__}")

import torch
import groundingdino
import segment_anything

print("torch cuda available =", torch.cuda.is_available())
print("groundingdino import ok")
print("segment_anything import ok")

from groundingdino.util.inference import Model
from segment_anything import sam_model_registry, SamPredictor

print("GroundingDINO Model import ok")
print("SAM import ok")


In [ ]:
# Cell 4: Download model checkpoints

%cd /content
!mkdir -p /content/models
!wget -q -O /content/models/groundingdino_swint_ogc.pth https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!wget -q -O /content/models/sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
!ls -lh /content/models


In [ ]:
# Cell 5: Mount Google Drive and configure paths

from google.colab import drive
from pathlib import Path
import os
import time

MOUNT_POINT = "/content/drive"

if os.path.ismount(MOUNT_POINT):
    print(f"Drive already mounted at {MOUNT_POINT}")
else:
    try:
        drive.mount(MOUNT_POINT, force_remount=True)
    except Exception as exc:
        print("First mount attempt failed:", repr(exc))
        print("Retrying in 3 seconds...")
        time.sleep(3)
        drive.mount(MOUNT_POINT, force_remount=True)

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/InspecSafe-V1')
DRIVE_ARCHIVE_PATH = DRIVE_PROJECT_ROOT / 'test_Annotations.tar'
EXTRACT_ROOT = Path('/content/inspecsafe_test_data')
DATASET_ROOT = EXTRACT_ROOT / 'DATA_PATH'
SPLIT = 'test'
ANNOTATIONS_ROOT = DATASET_ROOT / SPLIT / 'Annotations'
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'auto_label_outputs_test_only'

PROMPTS = [
    'person', 'open flame', 'fire hydrant', 'fire extinguisher',
    'safety helmet', 'electrical box', 'electronic control cabinet',
    'pipeline', 'valve', 'pressure gauge'
]

BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
MAX_IMAGES = 5

if not DRIVE_PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Drive project root not found: {DRIVE_PROJECT_ROOT}')
if not DRIVE_ARCHIVE_PATH.exists():
    raise FileNotFoundError(f'Archive not found: {DRIVE_ARCHIVE_PATH}')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'DRIVE_PROJECT_ROOT={DRIVE_PROJECT_ROOT}')
print(f'DRIVE_ARCHIVE_PATH={DRIVE_ARCHIVE_PATH}')
print(f'EXTRACT_ROOT={EXTRACT_ROOT}')
print(f'OUTPUT_ROOT={OUTPUT_ROOT}')


In [ ]:
# Cell 6: Extract test_Annotations.tar and verify folders

import tarfile
import shutil

LOCAL_SPLIT_ROOT = DATASET_ROOT / SPLIT

if ANNOTATIONS_ROOT.exists():
    print(f"Reusing extracted data: {ANNOTATIONS_ROOT}")
else:
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    LOCAL_SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {DRIVE_ARCHIVE_PATH} -> {LOCAL_SPLIT_ROOT}")
    with tarfile.open(DRIVE_ARCHIVE_PATH, 'r') as archive:
        archive.extractall(LOCAL_SPLIT_ROOT)
    print("Extraction finished")

normal_root = ANNOTATIONS_ROOT / 'Normal_data'
anomaly_root = ANNOTATIONS_ROOT / 'Anomaly_data'

if not ANNOTATIONS_ROOT.exists():
    raise FileNotFoundError(f'Annotations root not found after extraction: {ANNOTATIONS_ROOT}')

print(f'ANNOTATIONS_ROOT={ANNOTATIONS_ROOT}')
print(f'Normal exists={normal_root.exists()}')
print(f'Anomaly exists={anomaly_root.exists()}')


In [ ]:
# Test Cell 7: Count images and preview a few paths

def collect_rgb_images(annotations_root, max_images=None):
    image_paths = []
    for pattern in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(sorted(annotations_root.rglob(pattern)))
    image_paths = sorted(set(image_paths))
    return image_paths[:max_images] if max_images else image_paths

images = collect_rgb_images(ANNOTATIONS_ROOT, MAX_IMAGES)

if not images:
    raise RuntimeError(f'No RGB images found under {ANNOTATIONS_ROOT}')

print(f'Collected {len(images)} images from {ANNOTATIONS_ROOT}')
for path in images[:5]:
    print(path)


In [ ]:
# Cell 8: Load GroundingDINO and SAM

import sys
from pathlib import Path

segment_anything_repo = Path("/content/segment-anything")
groundingdino_repo = Path("/content/GroundingDINO")

if str(segment_anything_repo) not in sys.path:
    sys.path.insert(0, str(segment_anything_repo))
if str(groundingdino_repo) not in sys.path:
    sys.path.insert(0, str(groundingdino_repo))

import json
import cv2
import numpy as np
import torch
import transformers

if transformers.__version__ != "4.41.2":
    raise RuntimeError(f"Expected transformers 4.41.2 before loading models, got {transformers.__version__}")

from groundingdino.util.inference import Model
from segment_anything import sam_model_registry, SamPredictor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE={DEVICE}')
print(f'transformers version = {transformers.__version__}')

grounding_model = Model(
    model_config_path='/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',
    model_checkpoint_path='/content/models/groundingdino_swint_ogc.pth',
)

sam = sam_model_registry['vit_b'](checkpoint='/content/models/sam_vit_b_01ec64.pth')
sam.to(device=DEVICE)
sam_predictor = SamPredictor(sam)

print('Models loaded successfully')


In [ ]:
# Cell 9: Run inference on a few images

import traceback

def mask_to_polygon(mask):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    contour = max(contours, key=cv2.contourArea).squeeze(axis=1)
    if contour.ndim != 2 or len(contour) < 3:
        return []
    return contour.astype(float).tolist()

def run_single_image(image_path):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f'Failed to read image: {image_path}')

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    detections = grounding_model.predict_with_classes(
        image=image_rgb,
        classes=PROMPTS,
        box_threshold=BOX_THRESHOLD,
        text_threshold=TEXT_THRESHOLD,
    )

    sam_predictor.set_image(image_rgb)
    preds = []

    for box, score, class_id in zip(detections.xyxy, detections.confidence, detections.class_id):
        transformed_box = sam_predictor.transform.apply_boxes(
            np.array([box], dtype=np.float32),
            image_rgb.shape[:2]
        )

        masks, _, _ = sam_predictor.predict_torch(
            point_coords=None,
            point_labels=None,
            boxes=torch.tensor(transformed_box, dtype=torch.float32, device=DEVICE),
            multimask_output=False,
        )

        polygon = mask_to_polygon(masks[0, 0].detach().cpu().numpy().astype(np.uint8))

        if not polygon:
            x1, y1, x2, y2 = [float(v) for v in box.tolist()]
            polygon = [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]

        preds.append({
            'label': PROMPTS[int(class_id)],
            'score': float(score),
            'bbox': [float(v) for v in box.tolist()],
            'polygon': polygon,
        })

    return {
        'image_path': str(image_path),
        'width': int(image_rgb.shape[1]),
        'height': int(image_rgb.shape[0]),
        'predictions': preds,
    }

records = []
failures = []

for i, image_path in enumerate(images, start=1):
    try:
        record = run_single_image(image_path)
        records.append(record)
        print(f'[{i}/{len(images)}] {image_path.name}: {len(record["predictions"])} predictions')
    except Exception as exc:
        error_text = ''.join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        failures.append({
            'image_path': str(image_path),
            'error': str(exc),
            'traceback': error_text,
        })
        print(f'[{i}/{len(images)}] failed: {image_path.name}: {exc}')

print(f'Success={len(records)} | Failures={len(failures)}')


In [ ]:
# Cell 10: Export JSONL, COCO, error log, and summary

import json

def polygon_area(points):
    if len(points) < 3:
        return 0.0
    area = 0.0
    for idx, point in enumerate(points):
        nxt = points[(idx + 1) % len(points)]
        area += point[0] * nxt[1] - nxt[0] * point[1]
    return abs(area) / 2.0

def polygon_to_bbox(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]

def flatten_polygon(points):
    flat = []
    for x_coord, y_coord in points:
        flat.extend([x_coord, y_coord])
    return flat

def to_coco(records):
    categories = {}
    images_payload = []
    annotations_payload = []
    ann_id = 1

    for image_id, record in enumerate(records, start=1):
        images_payload.append({
            'id': image_id,
            'file_name': record['image_path'],
            'width': record['width'],
            'height': record['height'],
        })

        for pred in record['predictions']:
            categories.setdefault(pred['label'], len(categories) + 1)
            annotations_payload.append({
                'id': ann_id,
                'image_id': image_id,
                'category_id': categories[pred['label']],
                'bbox': polygon_to_bbox(pred['polygon']),
                'segmentation': [flatten_polygon(pred['polygon'])],
                'area': polygon_area(pred['polygon']),
                'iscrowd': 0,
                'score': pred.get('score'),
            })
            ann_id += 1

    categories_payload = [
        {'id': cid, 'name': name}
        for name, cid in sorted(categories.items(), key=lambda item: item[1])
    ]

    return {
        'images': images_payload,
        'annotations': annotations_payload,
        'categories': categories_payload,
    }

predictions_path = OUTPUT_ROOT / 'predictions.jsonl'
coco_path = OUTPUT_ROOT / 'predictions_coco.json'
failures_path = OUTPUT_ROOT / 'failures.json'
summary_path = OUTPUT_ROOT / 'run_summary.json'

with predictions_path.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

coco_payload = to_coco(records)
coco_path.write_text(json.dumps(coco_payload, indent=2, ensure_ascii=False), encoding='utf-8')
failures_path.write_text(json.dumps(failures, indent=2, ensure_ascii=False), encoding='utf-8')

summary = {
    'split': SPLIT,
    'max_images': MAX_IMAGES,
    'success_count': len(records),
    'failure_count': len(failures),
    'prediction_count': sum(len(record['predictions']) for record in records),
}

summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

print(predictions_path)
print(coco_path)
print(failures_path)
print(summary_path)
print(summary)
